In [1]:
!pip install pycocotools torchmetrics torchvision

import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import matplotlib.pyplot as plt
import numpy as np


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Environment ready. Training on: {device}")

Environment ready. Training on: cuda


In [2]:
import os
import random
import torch
import torchvision
import numpy as np
import torchvision.transforms.functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
from torch.utils.data import Dataset

class SegmentationTransforms:
    def __init__(self, is_train=True):
        self.is_train = is_train

    def __call__(self, image, target):
        if self.is_train:
            if random.random() > 0.5:
                image = F.adjust_brightness(image, brightness_factor=random.uniform(0.7, 1.3))
            if random.random() > 0.5:
                image = F.adjust_contrast(image, contrast_factor=random.uniform(0.7, 1.3))
            if random.random() > 0.5:
                image = F.adjust_saturation(image, saturation_factor=random.uniform(0.7, 1.3))
            if random.random() > 0.3:
                image = F.rgb_to_grayscale(image, num_output_channels=3)
        return image, target

class TeaBoxDataset(Dataset):
    def __init__(self, root_dir, annotation_file, transforms=None):
        self.root_dir = root_dir
        self.coco = COCO(annotation_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, index):
        img_id = self.ids[index]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        path = self.coco.loadImgs(img_id)[0]['file_name']
        img = torchvision.io.read_image(os.path.join(self.root_dir, path)).float() / 255.0
        
        masks = []
        boxes = []
        labels = []
        
        for ann in anns:
            masks.append(self.coco.annToMask(ann))
            xmin, ymin, w, h = ann['bbox']
            boxes.append([xmin, ymin, xmin + w, ymin + h])
            labels.append(ann['category_id'])

        target = {}
        target["boxes"] = torch.as_tensor(boxes, dtype=torch.float32)
        target["labels"] = torch.as_tensor(labels, dtype=torch.int64)
        target["masks"] = torch.as_tensor(np.array(masks), dtype=torch.uint8)
        target["image_id"] = torch.tensor([img_id])

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.ids)

def collate_fn(batch):
    return tuple(zip(*batch))


BATCH_SIZE = 2
LEARNING_RATE = 0.001
NUM_CLASSES = 2 

def get_mask_rcnn_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    return model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_mask_rcnn_model(NUM_CLASSES)
model.to(device)

optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], 
                            lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [3]:
import gc
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from tqdm import tqdm


TRAIN_IMG_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox/train"
TRAIN_JSON = "/kaggle/input/datasets/muhammadunssrahim/teabox/train/_annotations.coco.json"
VAL_IMG_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox/valid"
VAL_JSON = "/kaggle/input/datasets/muhammadunssrahim/teabox/valid/_annotations.coco.json"

NUM_EPOCHS = 30  


train_dataset = TeaBoxDataset(TRAIN_IMG_DIR, TRAIN_JSON, transforms=SegmentationTransforms(is_train=True))
val_dataset = TeaBoxDataset(VAL_IMG_DIR, VAL_JSON, transforms=SegmentationTransforms(is_train=False))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

print(f"\nStarting memory-optimized training pipeline for 'TeaBox found!' for {NUM_EPOCHS} epochs...")
print("=" * 80)


for epoch in range(1, NUM_EPOCHS + 1):
    
    model.train()
    total_loss = 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] Train", leave=False)
    
    for images, targets in train_loop:
    
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        train_loop.set_postfix(loss=losses.item())
        
        
        del images, targets, loss_dict, losses
        torch.cuda.empty_cache()
        
    lr_scheduler.step()
    avg_loss = total_loss / len(train_loader)
    
    
    model.eval()
    metric = MeanAveragePrecision(iou_type="segm")
    
    val_loop = tqdm(val_loader, desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] Valid", leave=False)
    
    with torch.no_grad():
        for images, targets in val_loop:
        
            images = list(img.to(device) for img in images)
            outputs = model(images)
            
            
            formatted_targets = [{
                "masks": t["masks"].cpu(),
                "labels": t["labels"].cpu()
            } for t in targets]
            
            formatted_preds = [{
                "masks": (out["masks"].squeeze(1) > 0.5).to(torch.uint8).cpu(), 
                "labels": out["labels"].cpu(),
                "scores": out["scores"].cpu()
            } for out in outputs]
            
            metric.update(formatted_preds, formatted_targets)
            
            # AGGRESSIVE MEMORY CLEARING: Free VRAM after every validation image
            del images, outputs, formatted_preds, formatted_targets
            torch.cuda.empty_cache()

    results = metric.compute()
    
    
    map_50 = results['map_50'].item()
    map_overall = results['map'].item()
    recall = results['mar_100'].item()
    
    
    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] | Loss: {avg_loss:.4f} | Val mAP@0.50: {map_50:.4f} | Val mAP@0.50:0.95: {map_overall:.4f} | Val Recall: {recall:.4f}")


    gc.collect()
    torch.cuda.empty_cache()

print("=" * 80)
print("Training completely finished!")


SAVE_PATH = "/kaggle/working/teabox_mask_rcnn.pth"
torch.save(model.state_dict(), SAVE_PATH)
print(f"\n✅ Model weights successfully saved to: {SAVE_PATH}")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!

Starting memory-optimized training pipeline for 'TeaBox found!' for 30 epochs...


Epoch [01/30] | Loss: 1.3235 | Val mAP@0.50: 0.0027 | Val mAP@0.50:0.95: 0.0009 | Val Recall: 0.0933


Epoch [02/30] | Loss: 0.7489 | Val mAP@0.50: 0.0086 | Val mAP@0.50:0.95: 0.0024 | Val Recall: 0.1000


Epoch [03/30] | Loss: 0.7246 | Val mAP@0.50: 0.0064 | Val mAP@0.50:0.95: 0.0014 | Val Recall: 0.0933


Epoch [04/30] | Loss: 0.6897 | Val mAP@0.50: 0.0107 | Val mAP@0.50:0.95: 0.0035 | Val Recall: 0.1467


...

Epoch [26/30] | Loss: 0.4822 | Val mAP@0.50: 0.1323 | Val mAP@0.50:0.95: 0.0932 | Val Recall: 0.2267


Epoch [27/30] | Loss: 0.4811 | Val mAP@0.50: 0.1323 | Val mAP@0.50:0.95: 0.0932 | Val Recall: 0.2267


Epoch [28/30] | Loss: 0.4843 | Val mAP@0.50: 0.1322 | Val mAP@0.50:0.95: 0.0932 | Val Recall: 0.2267


Epoch [29/30] | Loss: 0.4914 | Val mAP@0.50: 0.1322 | Val mAP@0.50:0.95: 0.0932 | Val Recall: 0.2267


Epoch [30/30] | Loss: 0.4781 | Val mAP@0.50: 0.1322 | Val mAP@0.50:0.95: 0.0932 | Val Recall: 0.2267
Training completely finished!

✅ Model weights successfully saved to: /kaggle/working/teabox_mask_rcnn.pth


In [4]:
import os
import zipfile
from IPython.display import FileLink, display

# 1. Define your file paths
pth_path = "/kaggle/working/teabox_mask_rcnn.pth"
zip_path = "teabox_model.zip" # Saving in the current working directory

# 2. Compress the weights into a zip file
print("Compressing model weights into a zip file...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # arcname ensures the zip just contains the file, not the whole folder structure
    zipf.write(pth_path, arcname="teabox_mask_rcnn.pth")

print("✅ Compression complete!")

# 3. Generate the clickable download link
print("\n👇 Click the link below to download your model weights:")
display(FileLink(zip_path))

Compressing model weights into a zip file...
✅ Compression complete!

👇 Click the link below to download your model weights:


/kaggle/working/teabox_model.zip